## About Guided Cursor

Welcome to **Guided Cursor**, an AI-powered platform designed to guide you through programming problems step by step. As with all AI systems, responses may not always be perfectly accurate. You are encouraged to think critically and verify all output independently.

**Privacy.** We may collect anonymised usage data to improve the platform and support academic research into AI-assisted pedagogy. No data will be shared outside the project team. Your usage and performance will **not** be disclosed to module leaders and will have **no bearing** on your academic grades.

**Data Retention.** This platform may be taken offline at the end of the academic term, and all stored data may be permanently deleted. Please back up any materials you wish to keep in advance.

**Contact.** For any questions or concerns, please reach out to **hello@guidedcursor.studio**.

# The Cooley-Tukey FFT Algorithm

This supplementary notebook accompanies *The Discrete Fourier Transform and Fast Fourier Transform*. In that notebook we implemented the DFT with a double loop and saw that it costs $\mathcal{O}(N^2)$ operations. Here we derive and implement the algorithm that reduces this to $\mathcal{O}(N \log N)$.

**Learning objectives.** By the end of this notebook you will be able to:

1. Derive the Cooley-Tukey decomposition from the DFT formula, step by step.
2. Explain the role of twiddle factors and why symmetry halves the work.
3. Implement a recursive radix-2 FFT from scratch.
4. Verify the implementation against NumPy and confirm $\mathcal{O}(N \log N)$ scaling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from fourier_hints import show_hint
from fourier_verify import check_twiddle_factors, check_cooley_tukey_fft

## Prerequisites

This notebook builds on *The Discrete Fourier Transform and Fast Fourier Transform*. We begin by recalling the DFT formula and introducing a shorthand that will keep the algebra compact.

The DFT of a signal $x[n]$ of length $N$ is

$$X[k] = \sum_{n=0}^{N-1} x[n] \, e^{-j\,2\pi\, k\, n \,/\, N}, \qquad k = 0, 1, \ldots, N-1$$

**Shorthand.** Define $W_N = e^{-j\,2\pi / N}$, the principal $N$-th root of unity. Then

$$X[k] = \sum_{n=0}^{N-1} x[n] \; W_N^{\,kn}$$

The naive implementation uses two nested loops, giving $\mathcal{O}(N^2)$ complexity. We now derive the $\mathcal{O}(N \log N)$ algorithm.

In [ ]:
## Complete code -- just run this cell.

def naive_dft(x):
    """Compute the DFT using two nested loops -- O(N^2)."""
    x = np.asarray(x, dtype=complex)
    N = len(x)
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += x[n] * np.exp(-2j * np.pi * k * n / N)
    return X

print("naive_dft loaded.")

## 1. The Key Idea: Splitting Even and Odd

The DFT sums over all $N$ samples. Can we break this sum into smaller pieces?

**Step 1.** Separate the sum into even-indexed terms ($n = 0, 2, 4, \ldots$) and odd-indexed terms ($n = 1, 3, 5, \ldots$):

$$X[k] = \sum_{\text{even } n} x[n]\, W_N^{kn} \;+\; \sum_{\text{odd } n} x[n]\, W_N^{kn}$$

**Step 2.** Substitute $n = 2m$ for the even terms and $n = 2m + 1$ for the odd terms, where $m = 0, 1, \ldots, N/2 - 1$:

$$X[k] = \sum_{m=0}^{N/2-1} x[2m]\, W_N^{k \cdot 2m} \;+\; \sum_{m=0}^{N/2-1} x[2m+1]\, W_N^{k(2m+1)}$$

**Step 3.** Simplify the exponents. This is the key algebraic step: $W_N^{2km} = e^{-j\,2\pi \cdot 2km/N} = e^{-j\,2\pi\, km/(N/2)} = W_{N/2}^{\,km}$.

> *Concrete example (N = 8):* $\;W_8^{\,2km} = e^{-j\,2\pi \cdot 2km/8} = e^{-j\,2\pi\, km/4} = W_4^{\,km}$.

For the odd sum, factor out $W_N^k$ from $W_N^{k(2m+1)} = W_N^{2km} \cdot W_N^k = W_{N/2}^{km} \cdot W_N^k$.

**Step 4.** The result:

$$\boxed{X[k] = \underbrace{\sum_{m=0}^{N/2-1} x[2m]\; W_{N/2}^{\,km}}_{E[k]} \;+\; W_N^k \;\underbrace{\sum_{m=0}^{N/2-1} x[2m+1]\; W_{N/2}^{\,km}}_{O[k]}}$$

Each half-sum is itself a DFT of length $N/2$:

- $E[k]$ is the DFT of the **even-indexed** samples $x[0], x[2], x[4], \ldots$
- $O[k]$ is the DFT of the **odd-indexed** samples $x[1], x[3], x[5], \ldots$

**One DFT of size $N$ has become two DFTs of size $N/2$, plus a simple combine step.**

In [ ]:
## Complete code -- just run this cell.

# An 8-point test signal.
x = np.array([1.0, -0.5, 0.8, 0.3, -0.2, 1.2, -0.7, 0.4])
N = len(x)

fig, ax = plt.subplots(figsize=(7, 3))

# Even indices (steelblue), odd indices (crimson).
even_idx = np.arange(0, N, 2)
odd_idx  = np.arange(1, N, 2)

markerline, stemlines, baseline = ax.stem(even_idx, x[even_idx], linefmt="-",
                                          markerfmt="o", basefmt=" ", label="Even")
plt.setp(stemlines, color="steelblue", linewidth=2)
plt.setp(markerline, color="steelblue", markersize=8)

markerline, stemlines, baseline = ax.stem(odd_idx, x[odd_idx], linefmt="-",
                                          markerfmt="s", basefmt=" ", label="Odd")
plt.setp(stemlines, color="crimson", linewidth=2)
plt.setp(markerline, color="crimson", markersize=8)

ax.set_xlabel("Sample index $n$", fontsize=13)
ax.set_ylabel("$x[n]$", fontsize=13)
ax.set_title("Even / Odd Index Split", fontsize=15)
ax.set_xticks(range(N))
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## Complete code -- just run this cell.

# Verify the even/odd decomposition numerically (N = 8).
X_direct = naive_dft(x)                        # Full DFT.

E = naive_dft(x[0::2])                          # DFT of even-indexed samples.
O = naive_dft(x[1::2])                          # DFT of odd-indexed samples.
W = np.exp(-2j * np.pi * np.arange(N) / N)      # Twiddle factors W_N^k.

X_split = np.zeros(N, dtype=complex)
for k in range(N):
    X_split[k] = E[k % (N // 2)] + W[k] * O[k % (N // 2)]

# Compare.
print(f"{'k':<4} {'Direct DFT':>20} {'E + W·O':>20}")
print("-" * 49)
for k in range(N):
    d = X_direct[k]
    s = X_split[k]
    print(f"{k:<5} {d.real:+9.4f}{d.imag:+9.4f}j  {s.real:+9.4f}{s.imag:+9.4f}j")

## 2. Twiddle Factors

The term $W_N^k = e^{-j\,2\pi\, k/N}$ that multiplies $O[k]$ before combining is called the **twiddle factor**. Each twiddle factor is a point on the unit circle in the complex plane, equally spaced at angles of $2\pi / N$.

For $N = 8$, we only need $W_8^0, W_8^1, W_8^2, W_8^3$ (that is, $N/2 = 4$ of them, as we shall see).

In [ ]:
## Complete code -- just run this cell.

N_tw = 8
half = N_tw // 2
angles = -2 * np.pi * np.arange(half) / N_tw
twiddles = np.exp(1j * angles)

fig, ax = plt.subplots(figsize=(5, 5))

# Unit circle.
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), color="grey", linewidth=0.8, alpha=0.5)
ax.axhline(0, color="grey", linewidth=0.5, alpha=0.4)
ax.axvline(0, color="grey", linewidth=0.5, alpha=0.4)

# Twiddle factors.
ax.scatter(twiddles.real, twiddles.imag, color="steelblue", s=100, zorder=5)
for k in range(half):
    ax.annotate(f"$W_8^{k}$",
                xy=(twiddles[k].real, twiddles[k].imag),
                xytext=(12, 10), textcoords="offset points",
                fontsize=13, color="steelblue")

ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.4, 1.4)
ax.set_aspect("equal")
ax.set_xlabel("Real", fontsize=13)
ax.set_ylabel("Imaginary", fontsize=13)
ax.set_title(f"Twiddle Factors for $N = {N_tw}$", fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. The Symmetry That Halves the Work

So far, the formula $X[k] = E[k] + W_N^k \cdot O[k]$ gives us each output bin. However, $E[k]$ and $O[k]$ are DFTs of length $N/2$, so they are **periodic** with period $N/2$:

$$E[k + N/2] = E[k], \qquad O[k + N/2] = O[k]$$

Meanwhile, the twiddle factor has a sign-flip symmetry:

$$W_N^{\,k + N/2} = e^{-j\,2\pi(k + N/2)/N} = e^{-j\,2\pi k/N} \cdot \underbrace{e^{-j\pi}}_{= \;-1} = -W_N^k$$

Putting these two facts together:

$$\boxed{\begin{aligned}
X[k] &= E[k] + W_N^k \cdot O[k] \\
X[k + N/2] &= E[k] - W_N^k \cdot O[k]
\end{aligned}} \qquad k = 0, 1, \ldots, N/2 - 1$$

Each pair $(E[k],\, O[k])$ produces **two** output values, $X[k]$ and $X[k + N/2]$, with just one multiplication and two additions. This is the **butterfly** operation.

In [ ]:
## Complete code -- just run this cell.

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(-0.2, 3.2)
ax.set_ylim(-0.5, 8.0)
ax.axis("off")
ax.set_title("Butterfly Stage ($N = 8$)", fontsize=15, pad=12)

half = 4
left_x, right_x = 0.4, 2.6

# Vertical positions: E[0..3] at top, O[0..3] below.
e_ys = [7.0, 6.0, 5.0, 4.0]
o_ys = [3.0, 2.0, 1.0, 0.0]
x_ys = [7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0, 0.0]

# Left labels.
for k in range(half):
    ax.text(left_x - 0.35, e_ys[k], f"$E[{k}]$", fontsize=12,
            ha="right", va="center", color="steelblue")
    ax.text(left_x - 0.35, o_ys[k], f"$O[{k}]$", fontsize=12,
            ha="right", va="center", color="crimson")

# Right labels.
for k in range(8):
    ax.text(right_x + 0.15, x_ys[k], f"$X[{k}]$", fontsize=12,
            ha="left", va="center")

# Draw butterfly connections.
for k in range(half):
    # E[k] -> X[k]  (+ branch)
    ax.annotate("", xy=(right_x, x_ys[k]), xytext=(left_x, e_ys[k]),
                arrowprops=dict(arrowstyle="->", color="steelblue",
                                linewidth=1.5, shrinkA=4, shrinkB=4))

    # E[k] -> X[k + N/2]  (+ branch)
    ax.annotate("", xy=(right_x, x_ys[k + half]), xytext=(left_x, e_ys[k]),
                arrowprops=dict(arrowstyle="->", color="steelblue",
                                linewidth=1.5, alpha=0.5, shrinkA=4, shrinkB=4))

    # O[k] -> X[k]  (twiddle branch)
    ax.annotate("", xy=(right_x, x_ys[k]), xytext=(left_x, o_ys[k]),
                arrowprops=dict(arrowstyle="->", color="crimson",
                                linewidth=1.5, shrinkA=4, shrinkB=4))

    # O[k] -> X[k + N/2]  (negative twiddle branch)
    ax.annotate("", xy=(right_x, x_ys[k + half]), xytext=(left_x, o_ys[k]),
                arrowprops=dict(arrowstyle="->", color="crimson",
                                linewidth=1.5, alpha=0.5, linestyle="--",
                                shrinkA=4, shrinkB=4))

    # Twiddle label.
    mid_y = (o_ys[k] + x_ys[k]) / 2
    ax.text(1.5, mid_y + 0.25, f"$W_8^{k}$", fontsize=11, ha="center",
            va="center", color="crimson",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# Legend.
ax.text(0.1, -0.4, "Solid = $+W_N^k \\cdot O[k]$", fontsize=10, color="crimson")
ax.text(1.7, -0.4, "Dashed = $-W_N^k \\cdot O[k]$", fontsize=10, color="crimson")

plt.tight_layout()
plt.show()

**Reading the diagram.** The left column contains the inputs and the right column contains the outputs.

- The **blue** values $E[0], \ldots, E[3]$ are the DFT of the even-indexed samples. The **red** values $O[0], \ldots, O[3]$ are the DFT of the odd-indexed samples.
- Each $E[k]$ feeds into two outputs: $X[k]$ in the top half and $X[k+4]$ in the bottom half (blue arrows).
- Each $O[k]$ is first multiplied by the twiddle factor $W_8^k$ and then also feeds into the same two outputs (red arrows).
- For the top-half outputs ($X[0]$ to $X[3]$), $E[k]$ and $W_8^k \cdot O[k]$ are **added** (solid red arrows).
- For the bottom-half outputs ($X[4]$ to $X[7]$), $E[k]$ and $W_8^k \cdot O[k]$ are **subtracted** (dashed red arrows).

The key point is that each pair $(E[k],\, O[k])$ requires only one complex multiplication (the twiddle factor multiply) to produce two output values. This is why the butterfly halves the work.

In [ ]:
## Complete code -- just run this cell.

# Numerical verification of the butterfly (same 8-point signal).
E = naive_dft(x[0::2])       # Length-4 DFT of even samples.
O = naive_dft(x[1::2])       # Length-4 DFT of odd samples.
W = np.exp(-2j * np.pi * np.arange(N // 2) / N)   # Only N/2 twiddle factors.

X_butterfly = np.zeros(N, dtype=complex)
X_butterfly[:N // 2] = E + W * O
X_butterfly[N // 2:] = E - W * O

X_ref = naive_dft(x)

print(f"{'k':<4} {'Butterfly':>20} {'Naive DFT':>20}")
print("-" * 49)
for k in range(N):
    b = X_butterfly[k]
    r = X_ref[k]
    print(f"{k:<5} {b.real:+9.4f}{b.imag:+9.4f}j  {r.real:+9.4f}{r.imag:+9.4f}j")

### Exercise 1: Twiddle Factors

Implement a function that returns the $N/2$ twiddle factors $W_N^k$ for $k = 0, 1, \ldots, N/2 - 1$.

In [ ]:
def twiddle_factors(N):
    """Return the N/2 twiddle factors W_N^k for k = 0, ..., N/2 - 1."""

    # ===== YOUR CODE BELOW =====
    return np.exp(-2j * np.pi * np.arange(N // 2) / N)
    # ===== YOUR CODE ABOVE =====

In [ ]:
show_hint("twiddle_factors")

In [ ]:
check_twiddle_factors(twiddle_factors)

## 4. Recursion: Divide and Conquer All the Way Down

We have shown that a DFT of size $N$ splits into two DFTs of size $N/2$. Each of those can be split again into two DFTs of size $N/4$, and so on, until we reach DFTs of size 1. A single-point DFT is trivial: $X[0] = x[0]$.

This requires $N$ to be a power of 2 (the **radix-2** restriction). The logic is the same as **merge sort**: split the problem in half, solve each half recursively, then combine the results.

In [ ]:
## Complete code -- just run this cell.

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis("off")
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-0.5, 4.0)
ax.set_title("Recursion Tree ($N = 8$)", fontsize=15, pad=10)

# Level positions (y) and node labels.
levels = {
    0: {"y": 3.5, "nodes": [(4.0, "DFT(8)")]},
    1: {"y": 2.5, "nodes": [(2.0, "DFT(4)"), (6.0, "DFT(4)")]},
    2: {"y": 1.5, "nodes": [(1.0, "DFT(2)"), (3.0, "DFT(2)"),
                              (5.0, "DFT(2)"), (7.0, "DFT(2)")]},
    3: {"y": 0.5, "nodes": [(0.5, "1"), (1.5, "1"), (2.5, "1"), (3.5, "1"),
                              (4.5, "1"), (5.5, "1"), (6.5, "1"), (7.5, "1")]},
}

bbox_style = dict(boxstyle="round,pad=0.3", fc="#e3f2fd", ec="steelblue", linewidth=1.2)
bbox_leaf  = dict(boxstyle="round,pad=0.25", fc="#fce4ec", ec="crimson", linewidth=1.2)

for lvl, info in levels.items():
    y = info["y"]
    for (cx, label) in info["nodes"]:
        style = bbox_leaf if lvl == 3 else bbox_style
        ax.text(cx, y, label, fontsize=11, ha="center", va="center", bbox=style)

# Edges.
edges = [
    (4.0, 3.5, 2.0, 2.5), (4.0, 3.5, 6.0, 2.5),
    (2.0, 2.5, 1.0, 1.5), (2.0, 2.5, 3.0, 1.5),
    (6.0, 2.5, 5.0, 1.5), (6.0, 2.5, 7.0, 1.5),
    (1.0, 1.5, 0.5, 0.5), (1.0, 1.5, 1.5, 0.5),
    (3.0, 1.5, 2.5, 0.5), (3.0, 1.5, 3.5, 0.5),
    (5.0, 1.5, 4.5, 0.5), (5.0, 1.5, 5.5, 0.5),
    (7.0, 1.5, 6.5, 0.5), (7.0, 1.5, 7.5, 0.5),
]
for (x1, y1, x2, y2) in edges:
    ax.plot([x1, x2], [y1 - 0.2, y2 + 0.2], color="grey", linewidth=1, alpha=0.6)

# Level annotations.
for lvl in range(4):
    ax.text(-0.4, levels[lvl]["y"], f"Level {lvl}", fontsize=10,
            ha="right", va="center", color="grey")

plt.tight_layout()
plt.show()

## 5. Why $\mathcal{O}(N \log N)$

At each level of the recursion tree, the butterfly combines across all sub-problems cost $\mathcal{O}(N)$ in total (every sample participates in exactly one butterfly). There are $\log_2 N$ levels, so the total cost is

$$\mathcal{O}(N \log N)$$

| $N$ | Naive $N^2$ | FFT $N \log_2 N$ | Speed-up |
|-----|------------|-------------------|----------|
| 1,024 | 1,048,576 | 10,240 | 102$\times$ |
| 65,536 | 4,294,967,296 | 1,048,576 | 4,096$\times$ |
| 1,000,000 | $10^{12}$ | $\approx 2 \times 10^7$ | $\approx$ 50,000$\times$ |

## 6. Implementation: Recursive Cooley-Tukey FFT

We now have everything we need. The algorithm, step by step:

1. **Base case.** If $N = 1$, return $x$ (a single-point DFT is just the value itself).
2. **Split.** Separate $x$ into even-indexed and odd-indexed elements.
3. **Recurse.** Compute $E$ = FFT of the even elements, $O$ = FFT of the odd elements.
4. **Twiddle.** Compute $T[k] = W_N^k \cdot O[k]$ for $k = 0, \ldots, N/2 - 1$.
5. **Butterfly.** Set $X[k] = E[k] + T[k]$ and $X[k + N/2] = E[k] - T[k]$.
6. **Return** $X$.

Implement this below. Assume `len(x)` is a power of 2.

In [ ]:
def cooley_tukey_fft(x):
    """Compute the DFT using the recursive Cooley-Tukey algorithm.

    Assumes len(x) is a power of 2.
    """
    x = np.asarray(x, dtype=complex)
    N = len(x)

    if N == 1:
        return x.copy()

    # ===== YOUR CODE BELOW =====
    E = cooley_tukey_fft(x[0::2])
    O = cooley_tukey_fft(x[1::2])
    T = np.exp(-2j * np.pi * np.arange(N // 2) / N) * O
    X = np.zeros(N, dtype=complex)
    X[:N // 2] = E + T
    X[N // 2:] = E - T
    return X
    # ===== YOUR CODE ABOVE =====

In [ ]:
show_hint("cooley_tukey_fft")

In [ ]:
check_cooley_tukey_fft(cooley_tukey_fft)

## 7. Verification

Let us confirm that our recursive FFT produces the same magnitude spectrum as NumPy's optimised implementation.

In [ ]:
## Complete code -- just run this cell.

# Test signal: three sinusoids (N = 64).
N_test = 64
n_test = np.arange(N_test)
sig = (np.sin(2 * np.pi * 3 * n_test / N_test)
     + 0.5 * np.sin(2 * np.pi * 7 * n_test / N_test)
     + 0.3 * np.sin(2 * np.pi * 15 * n_test / N_test))

X_ct  = cooley_tukey_fft(sig)
X_np  = np.fft.fft(sig)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)

for ax, X, title in zip(axes, [X_ct, X_np],
                         ["Cooley-Tukey FFT", "NumPy FFT"]):
    markerline, stemlines, baseline = ax.stem(
        np.arange(N_test // 2), np.abs(X[:N_test // 2]),
        linefmt="-", markerfmt="o", basefmt=" ")
    plt.setp(stemlines, color="steelblue", linewidth=1.5)
    plt.setp(markerline, color="steelblue", markersize=5)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Frequency bin $k$", fontsize=13)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("$|X[k]|$", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Results match: {np.allclose(X_ct, X_np)}")

## 8. Timing Comparison

Finally, we measure how the three implementations scale with $N$.

In [ ]:
## Complete code -- just run this cell.

sizes = [64, 128, 256, 512, 1024, 2048, 4096]

times_naive = []
times_ct    = []
times_np    = []

for sz in sizes:
    sig = np.random.randn(sz)

    if sz <= 1024:
        t0 = time.perf_counter()
        naive_dft(sig)
        times_naive.append(time.perf_counter() - t0)
    else:
        times_naive.append(np.nan)

    t0 = time.perf_counter()
    cooley_tukey_fft(sig)
    times_ct.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    np.fft.fft(sig)
    times_np.append(time.perf_counter() - t0)

# Table.
print(f"{'N':>6}  {'Naive (s)':>12}  {'CT-FFT (s)':>12}  {'NumPy (s)':>12}")
print("-" * 50)
for i, sz in enumerate(sizes):
    naive_str = f"{times_naive[i]:.6f}" if not np.isnan(times_naive[i]) else "..."
    print(f"{sz:>6}  {naive_str:>12}  {times_ct[i]:>12.6f}  {times_np[i]:>12.6f}")

# Log-log plot.
sizes_arr = np.array(sizes, dtype=float)

fig, ax = plt.subplots(figsize=(7, 4.5))

valid_naive = ~np.isnan(times_naive)
ax.plot(sizes_arr[valid_naive], np.array(times_naive)[valid_naive],
        "o-", color="crimson", linewidth=2, markersize=6, label="Naive DFT")
ax.plot(sizes_arr, times_ct,
        "s-", color="steelblue", linewidth=2, markersize=6, label="Cooley-Tukey FFT")
ax.plot(sizes_arr, times_np,
        "^-", color="forestgreen", linewidth=2, markersize=6, label="NumPy FFT")

# Reference lines.
ref_n2    = (sizes_arr / sizes_arr[0]) ** 2 * times_naive[0]
ref_nlogn = (sizes_arr * np.log2(sizes_arr)) / (sizes_arr[0] * np.log2(sizes_arr[0])) * times_ct[0]
ax.plot(sizes_arr, ref_n2, "--", color="grey", alpha=0.5, label="$O(N^2)$ ref.")
ax.plot(sizes_arr, ref_nlogn, ":", color="grey", alpha=0.5, label="$O(N \\log N)$ ref.")

ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("$N$", fontsize=13)
ax.set_ylabel("Time (s)", fontsize=13)
ax.set_title("Timing: Naive vs Cooley-Tukey vs NumPy", fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which="both")
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

The recursive Python FFT is much faster than the naive $\mathcal{O}(N^2)$ implementation and follows the $\mathcal{O}(N \log N)$ reference line. NumPy's FFT is faster still because it is written in optimised C, but the **scaling behaviour** is the same.

## Additional Notes

- **Iterative FFT.** Production implementations typically use a bottom-up (iterative) approach rather than recursion. This avoids function-call overhead and allows in-place computation.
- **Beyond radix-2.** More general algorithms (mixed-radix, split-radix, Bluestein) can handle arbitrary $N$, not just powers of 2.
- **The core insight is the same.** Every FFT variant exploits the symmetry of the complex exponential to decompose a large DFT into smaller sub-problems.

## Summary

| Step | What happens | Why it helps |
|------|-------------|-------------|
| Even/odd split | Separate the DFT sum into even-indexed and odd-indexed terms | Each half-sum is a DFT of half the size |
| Twiddle factors | $W_N^k$ rotates the odd-half result before combining | Connects the two half-DFTs into the full result |
| Butterfly symmetry | $W_N^{k+N/2} = -W_N^k$ | One pair $(E[k], O[k])$ gives two outputs, halving work |
| Recursion | Apply the same split all the way down to $N = 1$ | $\log_2 N$ levels $\times$ $\mathcal{O}(N)$ per level $=$ $\mathcal{O}(N \log N)$ |

**Key takeaway.** The Cooley-Tukey FFT computes *exactly the same result* as the naive DFT. It is faster because it avoids redundant computation by exploiting the symmetry of the complex exponential.